# Attma Router — LoRA SFT notebook

**Research-use only. Nothing here is trained, validated, clinically approved, or a diagnostic device. No real patient data, no real genomes, no real names.**

Question: can a LoRA fine-tune of Muse Glimmer 30B reliably route an Attma request to the right pipeline — and refuse or escalate biosecurity-sensitive requests — better than the base model?

This notebook follows the training pattern of the meta-hackathon-muse router notebook (build data locally from Hugging Face, train with the Fireworks Serverless Training API via the `fw-ai/cookbook` Muse Glimmer starter, deploy the tuned LoRA, evaluate, delete the deployment). It is an original Attma notebook, not a copy.

Caveat: the template result (route accuracy 22.5% → 90.0%) was measured on **n=40 held-out prompts** — a demo, not a paper. AttmaBench v1 requires n≥500 held-out prompts.

## 0. Setup

Needs `FIREWORKS_API_KEY` in the repo-root `.env`, and `firectl` installed for deployment. Training and deployment take too long to run live in a demo — run them ahead of time, confirm the deployment is ready, then present from the routing step. Delete the deployment afterwards; it bills per GPU-hour.

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent.parent
load_dotenv(ROOT / '.env')
assert os.environ.get('FIREWORKS_API_KEY'), 'Set FIREWORKS_API_KEY in repo-root .env'
print('FIREWORKS_API_KEY set:', bool(os.environ.get('FIREWORKS_API_KEY')))
print('ROOT:', ROOT)

## 1. Build the router data locally from Hugging Face

No dataset upload: build the training file locally from `iMartinDav/attma-router-v1`. Each row is `prompt` + strict-JSON `completion` with `route`, `complexity`, `confidence`, `reason`. Split by route and by disease area, and hold out at least 500 prompts for evaluation — never train on the held-out set.

In [ ]:
from datasets import load_dataset

ds = load_dataset('iMartinDav/attma-router-v1')
print(ds)
# Expected splits: train / validation / test (test = held-out, n>=500)
for split in ds:
    print(split, len(ds[split]))

In [ ]:
SYSTEM = (
    'You are the Attma Router. Route the request to exactly one pipeline. '
    'Reply with JSON only, on one line. Routes: germline, somatic, infectious, '
    'pgx, biosecurity_review, biosecurity_refuse, human_escalate. '
    'Research-use only. Never diagnose.'
)

def to_sft_row(ex):
    completion = json.dumps({
        'route': ex['route'],
        'complexity': ex['complexity'],
        'confidence': ex['confidence'],
        'reason': ex['reason'],
    }, ensure_ascii=False)
    return {'messages': [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': ex['prompt']},
        {'role': 'assistant', 'content': completion},
    ]}

train = ds['train'].map(to_sft_row, remove_columns=ds['train'].column_names)
out = Path('data/router_train.jsonl')
out.parent.mkdir(exist_ok=True)
with out.open('w') as f:
    for row in train:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print(f'wrote {len(train)} SFT rows to {out}')

## 2. LoRA SFT config for Muse Glimmer 30B

Base model: Muse Glimmer 30B. Method: LoRA SFT via the Fireworks Serverless Training API, following the `fw-ai/cookbook` Muse Glimmer starter. Suggested starting config (tune on validation, report the final config in the paper): LoRA rank 16, alpha 32, dropout 0.05, learning rate 1e-4, 2–3 epochs, batch size 8, max sequence length 2048. Do not train on the held-out test split.

In [ ]:
TRAIN_CONFIG = {
    'base_model': 'accounts/fireworks/models/muse-glimmer',
    'method': 'lora-sft',
    'lora_rank': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'learning_rate': 0.0001,
    'epochs': 3,
    'batch_size': 8,
    'max_seq_len': 2048,
    'train_file': 'data/router_train.jsonl',
}
print(json.dumps(TRAIN_CONFIG, indent=2))
# Submit the job with the fw-ai/cookbook starter (see its README for the exact
# firectl / API call for your Fireworks account), then record the job id here.
TRAIN_JOB_ID = '[FILL: Fireworks training job id]'
print('train job:', TRAIN_JOB_ID)

## 3. Deploy the tuned LoRA

Deploy with `firectl` (see the cookbook starter). Record the deployed model id and update `MODELS` in `run_router_bench.py` — the tuned entry currently points at the base model as a placeholder and must be replaced before a tuned run is meaningful.

In [ ]:
# Example (run in a terminal, not in this cell):
#   firectl deploy --model <tuned-lora-id> --name attma-router-v1
# Then set in run_router_bench.py:
#   MODELS[1]['model'] = 'accounts/fireworks/models/attma-router-v1'
DEPLOYED_MODEL_ID = '[FILL: accounts/fireworks/models/attma-router-v1]'
print('deployed model:', DEPLOYED_MODEL_ID)

## 4. Evaluate — base vs tuned, held-out only

Run the bench on the held-out test split (n≥500). The bench script grades deterministically (route accuracy + exact match) and writes resumable `results.jsonl`. Do not evaluate on training data and do not report training accuracy as a result.

In [ ]:
# In a terminal, from experiments/attma-router/:
#   python run_router_bench.py --n 500 --models base tuned
#   python run_router_bench.py --summarize-only
print('Run the bench from the terminal; see README.md for commands.')

## 5. Delete the deployment

Deployments bill per GPU-hour. Delete the deployment when evaluation is done, and record the total training + evaluation cost in the paper's reproducibility section. Pricing note from the template: LoRA SFT was free for jobs started before 2026-12-01 00:00 PT in that promotion — **do not assume that promotion still applies; check Fireworks pricing at run time and record what you actually paid.**

In [ ]:
# Example (run in a terminal, not in this cell):
#   firectl delete deployment attma-router-v1
print('Delete the deployment after evaluation; record total cost.')